# Step 06.4 — Context Window / Block Size

现在我们已经知道怎样把一段 `T+1` token 切成 `X(T)` 和 `Y(T)`。

下一个问题是：**一整本书可能有几十万甚至上百万 token，为什么不一次全部送进模型？**

这一节完整解决 5 个问题：

1. `N`、`block_size`、`T`、context 分别是什么；
2. 怎样从长语料切出一个训练窗口；
3. 为什么窗口可以重叠、边界本身没有语义；
4. causal 模型中每个位置真正拥有多少 context；
5. block size 对表达能力、显存/内存、计算量和推理有什么影响。

## 1. 先分清四个量：`N`、`block_size`、`T`、context

假设整份语料 tokenize 后有：

```text
N = 1,000,000 tokens
```

`N` 是整个 dataset / corpus 的总长度，它通常远大于一次模型输入。

我们人为选择：

```text
block_size = 64
```

意思是：**这个教学模型一次最多处理 64 个 token 位置。**

而 `T` 是某一次真正送进模型的实际 sequence length：

```text
T <= block_size
```

训练时为了效率，我们通常让：

```text
T = block_size
```

但推理刚开始时可能只有：

```text
T = 1, 2, 3, ...
```

直到达到最大窗口。

### `context` 不是另一个固定 shape

context 更接近“某个预测位置允许利用的历史信息”。

对于 causal 模型，在一个长度 `T=4` 的 block 中：

```text
position 0 -> context length 1
position 1 -> context length 2
position 2 -> context length 3
position 3 -> context length 4
```

所以：

> `block_size=64` 表示最大可用窗口，不代表 block 中每个位置天然都有 64 个历史 token。

## 2. 从长语料切出一个训练窗口

用一段很容易看的字符语料：

```text
abcdefghijklmnop
```

设：

```text
block_size = 4
```

为了得到 `X` 和 `Y` 各 4 个 token，我们必须先从语料拿：

```text
block_size + 1 = 5
```

个连续 token。

如果 `start=3`：

```text
corpus: a b c d e f g h i ...
              └───────┘
chunk:        d e f g h

X:            d e f g
Y:            e f g h
```

注意：`block_size` 指的是 X/Y 的长度，不是原始 chunk 的长度。

In [ ]:
import mlx.core as mx

corpus = "abcdefghijklmnop"
chars = list(corpus)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

data = mx.array([stoi[ch] for ch in corpus])

block_size = 4
start = 3

chunk = data[start : start + block_size + 1]
x = chunk[:-1]
y = chunk[1:]

decode = lambda ids: "".join(itos[int(i)] for i in ids)

print("corpus:", corpus)
print("chunk :", decode(chunk.tolist()), "shape =", chunk.shape)
print("X     :", decode(x.tolist()), "shape =", x.shape)
print("Y     :", decode(y.tolist()), "shape =", y.shape)

合法的 `start` 不能随便取到语料最后。

因为还要留出 `block_size+1` 个 token，所以最大起点满足：

```text
start + block_size < N
```

等价地：

```text
start <= N - block_size - 1
```

这个边界条件下一节写随机 batch sampler 时会真正用上。

## 3. Window 可以重叠：长语料不是被永久切成互不相干的小块

一个容易产生的误解是：

```text
整本书
↓
固定切成第 1 块、第 2 块、第 3 块
↓
边界永远不变
```

这不是我们教学 sampler 的基本思路。

同一条长序列可以从不同 start 位置抽取窗口：

```text
start 0: a b c d | target 到 e
start 1:   b c d e | target 到 f
start 2:     c d e f | target 到 g
start 3:       d e f g | target 到 h
```

它们大量重叠。

这样 token 不会永远被固定在某一个 block 边界位置，也能在不同上下文中反复成为训练目标。

### 用 coverage matrix 画出滑动窗口

这里介绍另一个常用绘图工具：`imshow()`。

它可以把二维矩阵直接画成格子图，非常适合：

- attention matrix
- confusion matrix
- mask
- window coverage

我们构造一个矩阵：行 = 不同 start，列 = corpus position；某窗口覆盖到的位置记为 1。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

N = len(corpus)
starts = list(range(6))
coverage = np.zeros((len(starts), N))

for row, s in enumerate(starts):
    coverage[row, s:s + block_size] = 1

fig, ax = plt.subplots(figsize=(10, 4))
ax.imshow(coverage, aspect="auto")
ax.set_xticks(range(N))
ax.set_xticklabels(list(corpus))
ax.set_yticks(range(len(starts)))
ax.set_yticklabels([f"start={s}" for s in starts])
ax.set_xlabel("corpus position / token")
ax.set_ylabel("training window")
ax.set_title("Overlapping context windows")
plt.show()

图里每一横行就是一个不同训练窗口。

你会看到窗口像尺子一样沿着 corpus 滑动，而且相邻窗口高度重叠。

这也意味着：**block 边界通常只是训练采样造成的工程边界，不一定对应句号、单词、段落等语义边界。**

## 4. 一个 block 内，每个位置真正拥有的 context 不一样

假设一个训练窗口：

```text
X = d e f g
Y = e f g h
```

结合 causal constraint，四个训练题其实是：

```text
d       -> e    context length = 1
d e     -> f    context length = 2
d e f   -> g    context length = 3
d e f g -> h    context length = 4
```

所以 `T=4` 代表这个 forward 有 4 个位置，不是每个位置都使用 4 个 token。

In [ ]:
x_text = "defg"
y_text = "efgh"

for t in range(len(x_text)):
    context = x_text[:t + 1]
    target = y_text[t]
    print(
        f"position={t}  "
        f"context={context!r:<6}  "
        f"context_length={len(context)}  "
        f"target={target!r}"
    )

### block 左边之外的 token 怎么办？

如果当前窗口从 `d` 开始，那么 corpus 中更早的：

```text
a b c
```

不在这个 forward 输入里，因此这个窗口中的模型**无法利用它们**。

这就是 context window 的真实限制：

> 模型无法在一次 forward 中直接依赖窗口之外的历史信息。

以后即使模型参数已经学到了语言规律，它当前 prediction 能动态参考的具体历史 token 仍然受 context window 限制。

## 5. 为什么不把 block_size 无限增大？——表达能力与计算成本的 trade-off

直觉上，context 越长越好，因为模型能参考更久以前的信息：

```text
短 context -> 只能看到附近信息
长 context -> 能利用更远的依赖
```

但长度不是免费的。

首先，仅 hidden representation：

```text
(B,T,C)
```

就会随着 `T` 线性增加内存。

更重要的是，我们以后学 Self-Attention 时会看到，每个位置要和其他位置建立关系，会形成：

```text
(T,T)
```

的 attention score matrix。

因此朴素 attention 的关键计算/中间存储会随 `T²` 增长。现在先不用推导，只看数量级：

In [ ]:
for T in [64, 128, 256, 512, 1024]:
    pair_count = T * T
    print(f"T={T:4d}  T^2={pair_count:10d} pairwise positions")

例如：

```text
T:  64 -> 128    长度 ×2
T²: 4096 -> 16384  pair 数 ×4
```

所以 context length 是一个典型 trade-off：

```text
更长历史信息
      vs
更多内存 / 计算 / 时间
```

### 在我们的 MiniGPT 里，`block_size` 还是一个架构上限

后面教学模型会使用 learned positional embedding，形状大致是：

```text
(block_size, C)
```

并构造最大 `block_size × block_size` 的 causal mask。

因此本项目里：

```text
T <= block_size
```

是实际硬限制之一。

推理时如果生成序列越来越长，最朴素的方法就是只保留最后 `block_size` 个 token 作为当前 context：

```text
very long generated history
              ↓ crop
[last block_size tokens]
              ↓ model
predict next token
```

更现代的长上下文方法、RoPE 扩展、KV cache 等以后再讲。

### 6.4 最终结论

- `N` 是完整语料长度，通常远大于单次输入；
- `block_size` 是本项目允许的最大 token window；
- `T` 是当前实际 sequence length，并满足 `T <= block_size`；
- 为产生长度 T 的 X/Y，需要先采样 `T+1` 个连续 token；
- 长语料通过不同 start 位置产生大量可重叠窗口，block 边界通常不是语义边界；
- causal block 内 position `t` 实际只拥有 `t+1` 个当前窗口内 token 的 context；
- 窗口之外的历史 token 本次 forward 无法直接使用；
- context 越长能利用更远信息，但内存/计算成本更高，Self-Attention 中尤其会出现 `T²` 级 pairwise 关系；
- 在我们的 learned-position MiniGPT 中，`block_size` 同时会成为一个实际最大 context 上限；
- 下一节 6.5：怎样随机选择多个 start position，一次构造 `(B,T)` 的训练 batch，并理解随机采样、seed 与 reproducibility。